In [ ]:
import numpy as np
import pickle
import sys
sys.path.append("..")
from scripts import indices

# Lateral field

In [ ]:
f = open('../Output/Determ/all_lat_nor.pckl', 'rb')
A_nor = pickle.load(f)
f.close()

In [ ]:
A_nor.keys()

### Efficiency

In [ ]:
ave_eff = {}

In [ ]:
for q in A_nor.keys():
    print(q)
    ave_eff[q] = {}
    for p in A_nor[q].keys():
        print(p)
        ave_eff[q][p] = {}
        for k in A_nor[q][p].keys():
            print(k)
            ave_eff[q][p][k] = {}
            for ind in A_nor[q][p][k].keys():
                ave_eff[q][p][k][ind] = indices.average_efficiency(A_nor[q][p][k][ind])

In [ ]:
fname = '../Output/Determ/alat_eff_stb.pckl'
f = open(fname, 'wb')
pickle.dump(ave_eff, f)
f.close()

### Connectivity and path lengths

In [ ]:
connected_pairs = {}

In [ ]:
for q in A_nor.keys():
    print(q)
    connected_pairs[q] = {}
    for p in A_nor[q].keys():
        print(p)
        connected_pairs[q][p] = {}
        for k in A_nor[q][p].keys():
            print(k)
            connected_pairs[q][p][k] = {}
            for ind in A_nor[q][p][k].keys():
                connected_pairs[q][p][k][ind], path_length = indices.path_length_connectedness(A_nor[q][p][k][ind])

In [ ]:
fname = '../Output/Determ/alat_cp_stb.pckl'
f = open(fname, 'wb')
pickle.dump(connected_pairs, f)
f.close()

### number of convergent and divergent hubs

In [ ]:
binary_flag = True; thresh = 15

In [ ]:
Hubs = {}
Hubs['in'] = {}; Hubs['out'] = {}

In [ ]:
for q in A_nor.keys():
    Hubs['in'][q] = {}
    Hubs['out'][q] = {}
    for p in A_nor[q].keys():
        Hubs['in'][q][p] = {}
        Hubs['out'][q][p] = {}
        for k in A_nor[q][p].keys():
            Hubs['in'][q][p][k] = {}
            Hubs['out'][q][p][k] = {}
            for ind in A_nor[q][p][k].keys():
                Hubs['in'][q][p][k][ind] = indices.hub_number(A_nor[q][p][k][ind],thresh,binary_flag, axisUsed=1)
                Hubs['out'][q][p][k][ind] = indices.hub_number(A_nor[q][p][k][ind],thresh,binary_flag, axisUsed=0)

In [ ]:
fname = '../Output/Determ/alat_hubs_'+str(thresh)+'_stb.pckl'
f = open(fname, 'wb')
pickle.dump(Hubs, f)
f.close()

### get cd-units

In [ ]:
fname = '../Output/Determ/alat_cp_stb.pckl'
f = open(fname, 'rb')
connected_pairs = pickle.load(f)
f.close()

cd_pairs = {}; thresh_list = [13, 14, 15]

In [ ]:
for thresh in thresh_list:
    cd_pairs[thresh] = {}
    for q in A_nor.keys():
        cd_pairs[thresh][q] = {}
        for p in A_nor[q].keys():
            cd_pairs[thresh][q][p] = {}
            for k in A_nor[q][p].keys():
                cd_pairs[thresh][q][p][k] = {}
                for ind in A_nor[q][p][k].keys():
                    cd_pairs[thresh][q][p][k][ind] = indices.cd_pairs(A_nor[q][p][k][ind], connected_pairs[q][p][k][ind], thresh)

In [ ]:
fname = '../Output/Determ/alat_cd_pairs_stb.pckl'
f = open(fname, 'wb')
pickle.dump(cd_pairs, f)
f.close()

### Stability of cd-units

In [ ]:
f = open('../Output/Determ/alat_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
num_config = {}; 
K = 10

for thresh in cd_pairs.keys():
    num_config[thresh] = {}
    for q in cd_pairs[thresh].keys():
        num_config[thresh][q] = {}
        for p_ind, p in enumerate(cd_pairs[thresh][q].keys()):
            num_config_p = np.zeros((K,31))
            for k in range(K):
                for i, step in enumerate(cd_pairs[thresh][q][p][k].keys()):
                    num_config_p[k, i] = cd_pairs[thresh][q][p][k][step].shape[1]
            num_config[thresh][q][p] = np.mean(num_config_p>0,0)

In [ ]:
for thresh in cd_pairs.keys():
    num_config[thresh] = {}
    for q in cd_pairs[thresh].keys():
        num_config[thresh][q] = np.zeros((2,len(cd_pairs[thresh][q].keys())))
        for p_ind, p in enumerate(cd_pairs[thresh][q].keys()):
            num_config_p = np.zeros(K)
            for k in cd_pairs[thresh][q][p].keys():
                num_config_temp = np.zeros(len(cd_pairs[thresh][q][p][k].keys()))
                for i, step in enumerate(cd_pairs[thresh][q][p][k].keys()):
                    num_config_temp[i] = cd_pairs[thresh][q][p][k][step].shape[1]
                num_config_p[k] = np.mean(num_config_temp==0)
            num_config[thresh][q][0,p_ind] = np.mean(num_config_p)
            num_config[thresh][q][1,p_ind] = np.std(num_config_p)            

In [ ]:
fname = '../Output/Determ/alat_num_config.pckl'
f = open(fname, 'wb')
pickle.dump(num_config, f)
f.close()

### number of units

In [ ]:
num_cd = {}; step = 15000

In [ ]:
for thresh in cd_pairs.keys():
    num_cd[thresh] = {}
    for q in cd_pairs[thresh].keys():
        num_cd[thresh][q] = np.zeros((2,len(cd_pairs[thresh][q].keys())))
        for p_ind, p in enumerate(cd_pairs[thresh][q].keys()):
            num_cd_p = np.zeros(K)
            for k in range(K):
                num_cd_p[k] = cd_pairs[thresh][q][p][k][step].shape[1]
            num_cd[thresh][q][0,p_ind] = np.mean(num_cd_p)
            num_cd[thresh][q][1,p_ind] = np.std(num_cd_p)

In [ ]:
fname = '../Output/Determ/alat_num_cd.pckl'
f = open(fname, 'wb')
pickle.dump(num_cd, f)
f.close()

### Number of source and target nodes, and their overlap

In [ ]:
f = open('../Output/Determ/alat_cp_stb.pckl', 'rb')
connected_pairs = pickle.load(f)
f.close()

f = open('../Output/Determ/alat_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
K = 10; step = 15000
num_targ = {}; num_sour = {}; prop_overlap = {}

In [ ]:
for thresh in thresh_list:
    num_targ[thresh] = {}
    num_sour[thresh] = {}
    prop_overlap[thresh] = {}
    for q in A_nor.keys():
        num_targ[thresh][q] = {}
        num_sour[thresh][q] = {}
        prop_overlap[thresh][q] = {}
        for p in A_nor[q].keys():
            num_targ[thresh][q][p] = np.zeros(K)
            num_sour[thresh][q][p] = np.zeros(K)
            prop_overlap[thresh][q][p] = np.zeros(K)
            for k in range(K):
                if cd_pairs[thresh][q][p][k][step].shape[1]>0:
                    num_targ_temp, num_sour_temp, prop_overlap_temp = indices.sour_targ_number(cd_pairs[thresh][q][p][k][step], connected_pairs[q][p][k][step])
                    num_targ[thresh][q][p][k] = np.mean(num_targ_temp)
                    num_sour[thresh][q][p][k] = np.mean(num_sour_temp)  
                    prop_overlap[thresh][q][p][k] = np.mean(prop_overlap_temp)
                else:
                    num_targ[thresh][q][p][k] = np.nan
                    num_sour[thresh][q][p][k] = np.nan
                    prop_overlap[thresh][q][p][k] = np.nan

In [ ]:
fname = '../Output/Determ/alat_num_sour.pckl'
f = open(fname, 'wb')
pickle.dump(num_sour, f)
f.close()

fname = '../Output/Determ/alat_num_targ.pckl'
f = open(fname, 'wb')
pickle.dump(num_targ, f)
f.close()

fname = '../Output/Determ/alat_prop_ovl.pckl'
f = open(fname, 'wb')
pickle.dump(prop_overlap, f)
f.close()

### Size and density of the intermediate subgraphs

In [ ]:
f = open('../Output/Determ/alat_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
K = 10; step = 15000
interm_size = {}; interm_density = {}; periph_density = {}

In [ ]:
for thresh in thresh_list:
    interm_size[thresh] = {}
    interm_density[thresh] = {}
    periph_density[thresh] = {}
    for q in A_nor.keys():
        interm_size[thresh][q] = {}
        interm_density[thresh][q] = {}
        periph_density[thresh][q] = {}        
        for p in A_nor[q].keys():
            interm_size[thresh][q][p] = {}
            interm_density[thresh][q][p] = {}
            periph_density[thresh][q][p] = {}
            for k in range(K):
                if cd_pairs[thresh][q][p][k][step].shape[1]>0:
                    interm_size[thresh][q][p][k],interm_density[thresh][q][p][k], periph_density[thresh][q][p][k] = indices.intermediate_subgraphs(A_nor[q][p][k][ind], cd_pairs[thresh][q][p][k][ind])
                else:
                    interm_size[thresh][q][p][k] = np.nan
                    interm_density[thresh][q][p][k] = np.nan
                    periph_density[thresh][q][p][k] = np.nan

In [ ]:
fname = '../Output/Determ/alat_interm_size.pckl'
f = open(fname, 'wb')
pickle.dump(interm_size, f)
f.close()

fname = '../Output/Determ/alat_interm_density.pckl'
f = open(fname, 'wb')
pickle.dump(interm_density, f)
f.close()

fname = '../Output/Determ/alat_periph_density.pckl'
f = open(fname, 'wb')
pickle.dump(periph_density, f)
f.close()

# Radial field

In [ ]:
f = open('../Output/Determ/trip_rad_1.pckl', 'rb')
A_nor = pickle.load(f)
f.close()

In [ ]:
A_nor.keys()

### Efficiency

In [ ]:
ave_eff = {}

In [ ]:
for q in A_nor.keys():
    print(q)
    ave_eff[q] = {}
    for p in A_nor[q].keys():
        print(p)
        ave_eff[q][p] = {}
        for k in A_nor[q][p].keys():
            print(k)
            ave_eff[q][p][k] = {}
            for ind in A_nor[q][p][k].keys():
                ave_eff[q][p][k][ind] = indices.average_efficiency(A_nor[q][p][k][ind])

In [ ]:
fname = '../Output/Determ/arad_eff_stb.pckl'
f = open(fname, 'wb')
pickle.dump(ave_eff, f)
f.close()

### Connectivity and path lengths

In [ ]:
connected_pairs = {}

In [ ]:
for q in A_nor.keys():
    print(q)
    connected_pairs[q] = {}
    for p in A_nor[q].keys():
        print(p)
        connected_pairs[q][p] = {}
        for k in A_nor[q][p].keys():
            print(k)
            connected_pairs[q][p][k] = {}
            for ind in A_nor[q][p][k].keys():
                connected_pairs[q][p][k][ind], path_length = indices.path_length_connectedness(A_nor[q][p][k][ind])

In [ ]:
fname = '../Output/Determ/arad_cp_stb.pckl'
f = open(fname, 'wb')
pickle.dump(connected_pairs, f)
f.close()

### number of convergent and divergent hubs

In [ ]:
binary_flag = True; thresh = 15

In [ ]:
Hubs = {}
Hubs['in'] = {}; Hubs['out'] = {}

In [ ]:
for q in A_nor.keys():
    Hubs['in'][q] = {}
    Hubs['out'][q] = {}
    for p in A_nor[q].keys():
        Hubs['in'][q][p] = {}
        Hubs['out'][q][p] = {}
        for k in A_nor[q][p].keys():
            Hubs['in'][q][p][k] = {}
            Hubs['out'][q][p][k] = {}
            for ind in A_nor[q][p][k].keys():
                Hubs['in'][q][p][k][ind] = indices.hub_number(A_nor[q][p][k][ind],thresh,binary_flag, axisUsed=1)
                Hubs['out'][q][p][k][ind] = indices.hub_number(A_nor[q][p][k][ind],thresh,binary_flag, axisUsed=0)

In [ ]:
fname = '../Output/Determ/arad_hubs_'+str(thresh)+'_stb.pckl'
f = open(fname, 'wb')
pickle.dump(Hubs, f)
f.close()

### get cd-units

In [ ]:
fname = '../Output/Determ/arad_cp_stb.pckl'
f = open(fname, 'rb')
connected_pairs = pickle.load(f)
f.close()

cd_pairs = {}; thresh_list = [13, 14, 15]

In [ ]:
for thresh in thresh_list:
    cd_pairs[thresh] = {}
    for q in A_nor.keys():
        cd_pairs[thresh][q] = {}
        for p in A_nor[q].keys():
            cd_pairs[thresh][q][p] = {}
            for k in A_nor[q][p].keys():
                cd_pairs[thresh][q][p][k] = {}
                for ind in A_nor[q][p][k].keys():
                    cd_pairs[thresh][q][p][k][ind] = indices.cd_pairs(A_nor[q][p][k][ind], connected_pairs[q][p][k][ind], thresh)

In [ ]:
fname = '../Output/Determ/arad_cd_pairs_stb.pckl'
f = open(fname, 'wb')
pickle.dump(cd_pairs, f)
f.close()

### Stability of cd-units

In [ ]:
f = open('../Output/Determ/arad_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
num_config = {}
K = 10

for thresh in cd_pairs.keys():
    num_config[thresh] = {}
    for q in cd_pairs[thresh].keys():
        num_config[thresh][q] = {}
        for p_ind, p in enumerate(cd_pairs[thresh][q].keys()):
            num_config_p = np.zeros((K,31))
            for k in range(K):
                for i, step in enumerate(cd_pairs[thresh][q][p][k].keys()):
                    num_config_p[k, i] = cd_pairs[thresh][q][p][k][step].shape[1]
            num_config[thresh][q][p] = np.mean(num_config_p>0,0)

In [ ]:
for thresh in cd_pairs.keys():
    num_config[thresh] = {}
    for q in cd_pairs[thresh].keys():
        num_config[thresh][q] = np.zeros((2,len(cd_pairs[thresh][q].keys())))
        for p_ind, p in enumerate(cd_pairs[thresh][q].keys()):
            num_config_p = np.zeros(K)
            for k in cd_pairs[thresh][q][p].keys():
                num_config_temp = np.zeros(len(cd_pairs[thresh][q][p][k].keys()))
                for i, step in enumerate(cd_pairs[thresh][q][p][k].keys()):
                    num_config_temp[i] = cd_pairs[thresh][q][p][k][step].shape[1]
                num_config_p[k] = np.mean(num_config_temp==0)
            num_config[thresh][q][0,p_ind] = np.mean(num_config_p)
            num_config[thresh][q][1,p_ind] = np.std(num_config_p)         

In [ ]:
fname = '../Output/Determ/arad_num_config.pckl'
f = open(fname, 'wb')
pickle.dump(num_config, f)
f.close()

### number of units

In [ ]:
num_cd = {}; step = 15000

In [ ]:
for thresh in cd_pairs.keys():
    num_cd[thresh] = {}
    for q in cd_pairs[thresh].keys():
        num_cd[thresh][q] = np.zeros((2,len(cd_pairs[thresh][q].keys())))
        for p_ind, p in enumerate(cd_pairs[thresh][q].keys()):
            num_cd_p = np.zeros(K)
            for k in range(K):
                num_cd_p[k] = cd_pairs[thresh][q][p][k][step].shape[1]
            num_cd[thresh][q][0,p_ind] = np.mean(num_cd_p)
            num_cd[thresh][q][1,p_ind] = np.std(num_cd_p)

In [ ]:
fname = '../Output/Determ/arad_num_cd.pckl'
f = open(fname, 'wb')
pickle.dump(num_cd, f)
f.close()

### Number of source and target nodes, and their overlap

In [ ]:
f = open('../Output/Determ/arad_cp_stb.pckl', 'rb')
connected_pairs = pickle.load(f)
f.close()

f = open('../Output/Determ/arad_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
K = 10; step = 15000
num_targ = {}; num_sour = {}; prop_overlap = {}

In [ ]:
for thresh in thresh_list:
    num_targ[thresh] = {}
    num_sour[thresh] = {}
    prop_overlap[thresh] = {}
    for q in A_nor.keys():
        num_targ[thresh][q] = {}
        num_sour[thresh][q] = {}
        prop_overlap[thresh][q] = {}
        for p in A_nor[q].keys():
            num_targ[thresh][q][p] = np.zeros(K)
            num_sour[thresh][q][p] = np.zeros(K)
            prop_overlap[thresh][q][p] = np.zeros(K)
            for k in range(K):
                if cd_pairs[thresh][q][p][k][step].shape[1]>0:
                    num_targ_temp, num_sour_temp, prop_overlap_temp = indices.sour_targ_number(cd_pairs[thresh][q][p][k][step], connected_pairs[q][p][k][step])
                    num_targ[thresh][q][p][k] = np.mean(num_targ_temp)
                    num_sour[thresh][q][p][k] = np.mean(num_sour_temp)  
                    prop_overlap[thresh][q][p][k] = np.mean(prop_overlap_temp)
                else:
                    num_targ[thresh][q][p][k] = np.nan
                    num_sour[thresh][q][p][k] = np.nan
                    prop_overlap[thresh][q][p][k] = np.nan

In [ ]:
fname = '../Output/Determ/arad_num_sour.pckl'
f = open(fname, 'wb')
pickle.dump(num_sour, f)
f.close()

fname = '../Output/Determ/arad_num_targ.pckl'
f = open(fname, 'wb')
pickle.dump(num_targ, f)
f.close()

fname = '../Output/Determ/arad_prop_ovl.pckl'
f = open(fname, 'wb')
pickle.dump(prop_overlap, f)
f.close()

### Size and density of the intermediate subgraphs

In [ ]:
f = open('../Output/Determ/arad_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
K = 10; step = 15000
interm_size = {}; interm_density = {}; periph_density = {}

In [ ]:
for thresh in thresh_list:
    interm_size[thresh] = {}
    interm_density[thresh] = {}
    periph_density[thresh] = {}
    for q in A_nor.keys():
        interm_size[thresh][q] = {}
        interm_density[thresh][q] = {}
        periph_density[thresh][q] = {}        
        for p in A_nor[q].keys():
            interm_size[thresh][q][p] = {}
            interm_density[thresh][q][p] = {}
            periph_density[thresh][q][p] = {}
            for k in range(K):
                if cd_pairs[thresh][q][p][k][step].shape[1]>0:
                    interm_size[thresh][q][p][k],interm_density[thresh][q][p][k], periph_density[thresh][q][p][k] = indices.intermediate_subgraphs(A_nor[q][p][k][ind], cd_pairs[thresh][q][p][k][ind])
                else:
                    interm_size[thresh][q][p][k] = np.nan
                    interm_density[thresh][q][p][k] = np.nan
                    periph_density[thresh][q][p][k] = np.nan

In [ ]:
fname = '../Output/Determ/arad_interm_size.pckl'
f = open(fname, 'wb')
pickle.dump(interm_size, f)
f.close()

fname = '../Output/Determ/arad_interm_density.pckl'
f = open(fname, 'wb')
pickle.dump(interm_density, f)
f.close()

fname = '../Output/Determ/arad_periph_density.pckl'
f = open(fname, 'wb')
pickle.dump(periph_density, f)
f.close()